# ARC Global Run Analysis

Loads merged results from an ARC quadrant sweep, provides summary statistics,
LCOA heatmap, cost-component breakdowns, and capacity mix analysis.

Currency is auto-detected from the `currency` column in the results CSV — all
cost column references are built dynamically so the notebook works with any
tech config (EUR, USD, AUD, etc.).

In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path('__file__').resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(NOTEBOOK_DIR)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model import plot_global_heatmap
importlib.reload(plot_global_heatmap)

print('Repo root:', repo_root)

## 1. Load results

In [ ]:
# ----- CONFIG -----
# Label for this finance scenario -- used as the run output folder name.
# Change this whenever you change financial inputs so scenarios stay separate.
RUN_LABEL = 'dea_2030_flat'

# All outputs for this scenario (results CSV, heatmap HTML, figure PNGs) live
# together under results/<RUN_LABEL>/ so each scenario is self-contained.
RUN_DIR      = repo_root / 'results' / RUN_LABEL
RESULTS_CSV  = RUN_DIR / 'global_run_results_1h_2030.csv'
HEATMAP_HTML = RUN_DIR / 'lcoa_heatmap.html'

COLOR_COLUMN = 'lcoa_eur_per_t'
COLOR_SCALE = 'Viridis_r'
CELL_SIZE_DEG = 1.0

# ----- FIGURE EXPORT CONFIG -----
# Figures are saved alongside the results CSV in results/<RUN_LABEL>/.
FIG_DIR = RUN_DIR
FIG_SCALE = 3

# ----- ANALYSIS FILTER -----
LCOA_CLIP_MAX = 2_000  # EUR/t

# ----- LOAD -----
df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df):,} locations from {RESULTS_CSV.name}')
print(f'Run folder: results/{RUN_LABEL}/')
print(f'Columns: {len(df.columns)}')
print(f'Lat range: {df.latitude.min():.0f} to {df.latitude.max():.0f}')
print(f'Lon range: {df.longitude.min():.0f} to {df.longitude.max():.0f}')
print(f'Countries: {df.country.nunique()}')

# ----- COST FILTER -----
if 'lcoa_eur_per_t' in df.columns:
    _cost_mask = df['lcoa_eur_per_t'] <= LCOA_CLIP_MAX
    dff = df[_cost_mask].copy()
    n_above_clip = int((~_cost_mask).sum())
else:
    dff = df.copy()
    n_above_clip = 0

if 'grid_energy_mwh' in df.columns:
    n_backstop = int((df['grid_energy_mwh'] >= 1_000).sum())
else:
    n_backstop = 0

print()
print(f'LCOA_CLIP_MAX = {LCOA_CLIP_MAX:,} EUR/t')
print(f'  Within clip (dff): {len(dff):,}  ({len(dff)/len(df)*100:.1f}%)')
print(f'  Above clip (heatmap only): {n_above_clip:,}')
print(f'  Used grid backstop (>1000 MWh/yr): {n_backstop:,}')


In [ ]:
def save_fig(
    fig,
    name: str,
    width: int = 900,
    height: int | None = None,
    scale: int | None = None,
) -> None:
    """Save fig to results/<RUN_LABEL>/. Requires kaleido."""
    import kaleido  # noqa: F401
    if height is None:
        height = round(width * 9 / 16)
    if scale is None:
        scale = FIG_SCALE
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out = FIG_DIR / f'{name}.png'
    fig.write_image(str(out), width=width, height=height, scale=scale)
    print(f'  ok  {out.relative_to(repo_root)}  ({width*scale}x{height*scale} px)')

print(f'save_fig() ready -- output dir: results/{RUN_LABEL}/')
print(f'Export scale: {FIG_SCALE}x')


## 2. Results header & summary statistics

In [ ]:
# Show first rows
display(df.head(10))

In [ ]:
# Key numeric summary
summary_cols = [
    'lcoa_eur_per_t',
    'total_cost_eur_per_year',
    'annual_ammonia_production_t',
    'wind_mw', 'solar_mw',
    'electrolysis_mw', 'ammonia_synthesis_mw',
    'compressed_hydrogen_store_mwh', 'battery_pcs_mw',
    'hydrogen_storage_capacity_t',
]
summary_cols = [c for c in summary_cols if c in df.columns]
display(dff[summary_cols].describe().round(2))

In [ ]:
# Top 20 cheapest locations
top20 = dff.nsmallest(20, 'lcoa_eur_per_t')[[
    'latitude', 'longitude', 'country', 'lcoa_eur_per_t',
    'wind_mw', 'solar_mw', 'electrolysis_mw', 'ammonia_synthesis_mw',
]].copy().reset_index(drop=True)
top20['country'] = top20['country'].fillna('Ocean/Unclaimed')
top20.index = top20.index + 1
top20.index.name = 'rank'
display(top20)


In [ ]:
# Country-level summary: median LCOA and location count
country_stats = (
    dff.groupby('country')
    .agg(
        n_locations=('lcoa_eur_per_t', 'count'),
        lcoa_median=('lcoa_eur_per_t', 'median'),
        lcoa_min=('lcoa_eur_per_t', 'min'),
        lcoa_max=('lcoa_eur_per_t', 'max'),
    )
    .sort_values('lcoa_median')
)
print(f'Countries with results: {len(country_stats)}')
display(country_stats.head(20).round(1))

## 3. LCOA heatmap

In [ ]:
# Reload module to pick up range_color / percentile_clip support
importlib.reload(plot_global_heatmap)

n_solved = len(df)
n_land_cells = len(pd.read_csv(repo_root / 'data' / '20251222_max_capacities.csv'))

# Heatmap uses ALL solved cells (df), not dff.
# Colour scale is clipped to [p2 of within-clip cells, LCOA_CLIP_MAX] so:
#   - cheap cells have full colour resolution across the interesting range
#   - expensive / infeasible cells all pile up at the top colour (dark = expensive)
#   - identical philosophy to the paper's cap at 1200 USD/t
_p02     = float(dff[COLOR_COLUMN].quantile(0.02))   # lower bound: strip cheapest 2% outliers
_clip_lo = _p02
_clip_hi = float(LCOA_CLIP_MAX)

fig_heatmap = plot_global_heatmap.plot_lcoa_heatmap(
    df,                          # <-- ALL cells, not just dff
    color_column=COLOR_COLUMN,
    cell_size_deg=CELL_SIZE_DEG,
    color_scale=COLOR_SCALE,
    range_color=(_clip_lo, _clip_hi),
)
fig_heatmap.update_layout(
    title=(
        f'Global LCOA (EUR/t NH₃) – DEA 2030 flat-finance sweep<br>'
        f'<sup>Colour scale {_clip_lo:.0f}–{_clip_hi:.0f} EUR/t NH\u2083 · '
        f'cells above {LCOA_CLIP_MAX:,} EUR/t shown at maximum colour</sup>'
    ),
    height=720,
)
fig_heatmap.show()
fig_heatmap.write_html(HEATMAP_HTML)
print(f'Saved heatmap: {HEATMAP_HTML.name}')
print(f'Colour range: {_clip_lo:.0f} – {_clip_hi:.0f} EUR/t')
print(f'White gaps: {n_land_cells - n_solved:,}  (solver failures / no weather data)')
print(f'Cells shown at max colour (>{LCOA_CLIP_MAX:,} EUR/t): {n_above_clip:,}')

In [ ]:
# Matplotlib heatmap — publication version with lat/lon cornice
importlib.reload(plot_global_heatmap)
import matplotlib.pyplot as _plt
_fig_mpl, _ = plot_global_heatmap.plot_lcoa_heatmap_mpl(
    df, color_column=COLOR_COLUMN, cell_size_deg=CELL_SIZE_DEG,
    range_color=(_clip_lo, _clip_hi),
)
_plt.show()
_plt.close(_fig_mpl)


## 4. LCOA distribution

In [ ]:
fig_hist = px.histogram(
    dff, x='lcoa_eur_per_t', nbins=80,
    title='Distribution of LCOA across all locations',
    labels={'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)'},
    color_discrete_sequence=['steelblue'],
)
fig_hist.add_vline(x=dff['lcoa_eur_per_t'].median(), line_dash='dash', line_color='red',
                   annotation_text=f"Median: {dff['lcoa_eur_per_t'].median():.0f}")
fig_hist.update_layout(height=400)
fig_hist.show()

## 5. Cost component breakdown

In [ ]:
# Identify LCOA component columns
component_cols = [c for c in df.columns if c.startswith('lcoa_component_') and c.endswith('_eur_per_t')]
component_labels = [
    c.replace('lcoa_component_', '').replace('_eur_per_t', '').replace('_', ' ').title()
    for c in component_cols
]

# Global mean cost breakdown
mean_components = dff[component_cols].mean().values

fig_pie = go.Figure(data=[go.Pie(
    labels=component_labels,
    values=mean_components,
    textinfo='label+percent',
    hovertemplate='%{label}: €%{value:.0f}/t<extra></extra>',
    hole=0.35,
)])
fig_pie.update_layout(
    title='Mean LCOA component breakdown (all locations)',
    height=500,
)
fig_pie.show()

In [ ]:
# Stacked bar: LCOA components for the 15 cheapest locations
top15 = dff.nsmallest(15, 'lcoa_eur_per_t').copy()
# Fill blank country (ocean / outside country polygons) with a human-readable label
top15['country_label'] = top15['country'].fillna('Ocean/Unclaimed')
top15['location'] = top15.apply(
    lambda r: f"{r['country_label']} ({r['latitude']:.0f}, {r['longitude']:.0f})", axis=1
)

fig_stack = go.Figure()
for col, label in zip(component_cols, component_labels):
    fig_stack.add_trace(go.Bar(
        x=top15['location'],
        y=top15[col],
        name=label,
        hovertemplate=f'{label}: €%{{y:.0f}}/t<extra></extra>',
    ))
fig_stack.update_layout(
    barmode='stack',
    title='LCOA breakdown – 15 cheapest locations',
    yaxis_title='LCOA (EUR/t NH₃)',
    xaxis_title='',
    height=500,
    xaxis_tickangle=-45,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig_stack.show()


## 6. Capacity mix analysis

In [ ]:
# Wind vs Solar capacity coloured by LCOA
fig_scatter = px.scatter(
    dff, x='wind_mw', y='solar_mw',
    color='lcoa_eur_per_t',
    color_continuous_scale='Cividis',
    hover_data=['country', 'latitude', 'longitude', 'electrolysis_mw'],
    title='Wind vs Solar capacity (coloured by LCOA)',
    labels={
        'wind_mw': 'Wind (MW)',
        'solar_mw': 'Solar (MW)',
        'lcoa_eur_per_t': 'LCOA (EUR/t)',
    },
    opacity=0.5,
)
fig_scatter.update_layout(height=500)
fig_scatter.show()

In [ ]:
# Hydrogen storage vs LCOA
df_h2 = dff.copy()
df_h2['country_label'] = df_h2['country'].fillna('Ocean/Unclaimed')
fig_h2 = px.scatter(
    df_h2, x='hydrogen_storage_capacity_t', y='lcoa_eur_per_t',
    color='country_label',
    hover_data=['latitude', 'longitude', 'wind_mw', 'solar_mw'],
    title='H₂ storage capacity vs LCOA',
    labels={
        'hydrogen_storage_capacity_t': 'H₂ storage (t)',
        'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)',
        'country_label': 'Country',
    },
    opacity=0.5,
)
fig_h2.update_layout(height=500, showlegend=False)
fig_h2.show()


## 7. Country box plots

In [ ]:
# Show top-20 countries by number of locations, sorted by median LCOA
top_countries = dff.country.value_counts().head(20).index.tolist()
df_top = dff[dff.country.isin(top_countries)].copy()

# Sort by median LCOA
country_order = (
    df_top.groupby('country')['lcoa_eur_per_t']
    .median()
    .sort_values()
    .index.tolist()
)

fig_box = px.box(
    df_top, x='country', y='lcoa_eur_per_t',
    category_orders={'country': country_order},
    title='LCOA distribution – top 20 countries by location count',
    labels={'lcoa_eur_per_t': 'LCOA (EUR/t NH₃)', 'country': ''},
    color='country',
)
fig_box.update_layout(
    height=500,
    showlegend=False,
    xaxis_tickangle=-45,
)
fig_box.show()

## 8. Coverage & data quality

In [ ]:
# Check for NaN/missing patterns
nan_counts = df.isna().sum()
nan_nonzero = nan_counts[nan_counts > 0]
if len(nan_nonzero) > 0:
    print(f'{len(nan_nonzero)} columns have missing values:')
    display(nan_nonzero.sort_values(ascending=False))
else:
    print('No missing values in any column.')

# ── Coverage stats from land CSV ──────────────────────────────────────────────
land_df = pd.read_csv(repo_root / 'data' / '20251222_max_capacities.csv')
n_land_cells = len(land_df)
n_solved    = len(df)
n_missing   = n_land_cells - n_solved

n_with_country  = df['country'].notna().sum()
n_ocean         = df['country'].isna().sum()

print()
print(f'Land CSV total cells  : {n_land_cells:,}')
print(f'Solved (in results)   : {n_solved:,}  ({n_solved/n_land_cells*100:.1f}%)')
print(f'Failed / skipped      : {n_missing:,}  ({n_missing/n_land_cells*100:.1f}%)')
print()
print(f'Solved with country   : {n_with_country:,}  (onshore / coastal land cells)')
print(f'Solved, no country    : {n_ocean:,}  (ocean cells or cells outside all country polygons)')
print()
print(f'--- Cost filter (dff, used in analytical plots) ---')
print(f'LCOA_CLIP_MAX         : {LCOA_CLIP_MAX:,} EUR/t')
print(f'Within clip (dff)     : {len(dff):,}  ({len(dff)/n_land_cells*100:.1f}% of all land cells)')
print(f'Above clip            : {n_above_clip:,}  (on heatmap at max colour; excluded from analysis)')
print()
print(f'--- Grid backstop (informational) ---')
print(f'Used backstop >1000 MWh/yr : {n_backstop:,}')
_backstop_in_dff = int((dff['grid_energy_mwh'] >= 1_000).sum()) if 'grid_energy_mwh' in dff.columns else 0
print(f'Of which within LCOA clip  : {_backstop_in_dff:,}  (included in analysis — low grid draw)')

In [ ]:
# Lat/Lon grid coverage scatter — ALL solved cells, colour clipped at LCOA_CLIP_MAX
# (same logic as heatmap: expensive/infeasible cells appear at max colour, not hidden)
lcoa_col = 'lcoa_eur_per_t'
_p02_all = float(df[lcoa_col].quantile(0.02))

_hover = ['country', lcoa_col, 'grid_energy_mwh'] if 'grid_energy_mwh' in df.columns else ['country', lcoa_col]
fig_grid = px.scatter(
    df, x='longitude', y='latitude',
    color=lcoa_col,
    color_continuous_scale='Cividis',
    range_color=(_p02_all, LCOA_CLIP_MAX),
    title=(
        f'All solved cells ({n_solved:,} locations, '
        f'{n_land_cells - n_solved:,} failed/unsolved) — '
        f'LCOA colour clipped to {_p02_all:.0f}–{LCOA_CLIP_MAX:,} EUR/t'
    ),
    labels={'longitude': 'Longitude', 'latitude': 'Latitude', lcoa_col: 'LCOA (EUR/t)'},
    opacity=0.6,
    hover_data=_hover,
)
fig_grid.update_layout(height=450, width=950)
fig_grid.update_traces(marker_size=3)
fig_grid.show()

print(f'LCOA range (all cells): {df[lcoa_col].min():.0f} – {df[lcoa_col].max():.0f} EUR/t')
print(f'Colour scale clipped to: {_p02_all:.0f} – {LCOA_CLIP_MAX:,} EUR/t')
print(f'Cells above cap (at max colour): {n_above_clip:,}  |  Unsolved: {n_land_cells - n_solved:,}')
print(f'Used grid backstop >1000 MWh/yr: {n_backstop:,}')

## 9. Export figures for paper

In [ ]:
print(f'Exporting -> {FIG_DIR.relative_to(repo_root)}/')
print()
# Matplotlib export for heatmap — publication quality with lat/lon cornice
_fig_mpl, _ = plot_global_heatmap.plot_lcoa_heatmap_mpl(
    df, color_column=COLOR_COLUMN, cell_size_deg=CELL_SIZE_DEG,
    range_color=(_clip_lo, _clip_hi),
)
_mpl_out = FIG_DIR / 'lcoa_heatmap.png'
FIG_DIR.mkdir(parents=True, exist_ok=True)
_fig_mpl.savefig(str(_mpl_out), dpi=150, bbox_inches='tight')
import matplotlib.pyplot as _plt; _plt.close(_fig_mpl)
print(f'  ok  {_mpl_out.relative_to(repo_root)}')
save_fig(fig_hist,    'lcoa_distribution',     width=900,  height=480)
save_fig(fig_pie,     'cost_components_mean',  width=750,  height=650)
save_fig(fig_stack,   'cost_breakdown_top15',  width=1100, height=580)
save_fig(fig_scatter, 'wind_solar_scatter',    width=900,  height=600)
save_fig(fig_h2,      'h2_storage_vs_lcoa',    width=900,  height=600)
save_fig(fig_box,     'country_lcoa_boxplots', width=1300, height=580)
save_fig(fig_grid,    'solved_grid_coverage',  width=1400, height=580)
print()
print(f'Done. {len(list(FIG_DIR.glob(chr(42)+".png")))} PNG files in {FIG_DIR.relative_to(repo_root)}/')
